In [3]:
import pydicom
import dicom2nifti
from pathlib import Path

# Percorsi
dicom_study_dir = Path("C:\\Users\\dosimetria4\\Naiara_gdr\\data\\prove\\anonymized_Dataset\\NCT07132658_5\\T0")
fixed_dir = dicom_study_dir.parent / "T0_fixed"
output_nifti_dir = dicom_study_dir.parent / "T0_Nifti_Out"

fixed_dir.mkdir(exist_ok=True)
output_nifti_dir.mkdir(exist_ok=True)

print("Iniettando la firma ufficiale DICOM nei file...")
# Correggiamo i file
for f in dicom_study_dir.glob("*"):
    if f.is_file():
        ds = pydicom.dcmread(f, force=True)

        # --- 1. IL FIX DEL FILE_META ---
        # Creiamo la "busta" se non esiste
        if not hasattr(ds, 'file_meta'):
            ds.file_meta = pydicom.dataset.FileMetaDataset()
            
        # Ripeschiamo il TransferSyntaxUID dal corpo del file (il tag DICOM è 0x0002, 0x0010)
        if (0x0002, 0x0010) in ds:
            ds.file_meta.TransferSyntaxUID = ds[0x0002, 0x0010].value
        elif not hasattr(ds.file_meta, 'TransferSyntaxUID'):
            # Caso limite disperato
            ds.file_meta.TransferSyntaxUID = pydicom.uid.ImplicitVRLittleEndian

        ds.decompress()

        # IL TRUCCO MAGICO: Creiamo i 128 byte vuoti. pydicom aggiungerà automaticamente "DICM" dopo di questi!
        ds.preamble = b"\0" * 128 
        ds.save_as(fixed_dir / f.name)

Iniettando la firma ufficiale DICOM nei file...


In [4]:
print("Provo la conversione in NIfTI sui file corretti...")
try:
    dicom2nifti.convert_directory(
        str(fixed_dir), 
        str(output_nifti_dir), 
        compression=True, 
        reorient=True
    )
    print(f"✅ Conversione completata! Controlla la cartella: {output_nifti_dir}")
except Exception as e:
    print(f"❌ Errore: {e}")

Provo la conversione in NIfTI sui file corretti...
✅ Conversione completata! Controlla la cartella: C:\Users\dosimetria4\Naiara_gdr\data\prove\anonymized_Dataset\NCT07132658_5\T0_Nifti_Out
